**Standard continued fractions.**
Any real number $x$ can be written as a **continued fraction** of the form:

$$
x = a_0 + \cfrac{1}{a_1 + \cfrac{1}{a_2 + \cfrac{1}{a_3 + \cdots}}}
$$

where each $a_k$ is a non-negative integer called a **partial quotient**,
written compactly as $[a_0; a_1, a_2, a_3, \ldots]$.
The algorithm to find the partial quotients is simple: take the integer
part, subtract it, then take the reciprocal of the remainder and repeat.

Continued fractions reveal deep structure in real numbers:
rational numbers produce **finite** continued fractions,
quadratic irrationals (like $\sqrt{2}$) produce **periodic** continued fractions,
and transcendental numbers like $e$ produce continued fractions with
recognizable but non-repeating patterns.

In [ ]:
"""std_cf.ipynb"""

# Cell 01 - Encode a real number as a continued fraction

import math


def encode_cf(x: float) -> list[int]:
    """Return the continued fraction representation of x as a list of partial quotients.

    Stops after 20 terms (for irrational numbers) or when the fractional
    part falls below 1e-11 (for rationals and floating-point approximations).
    The result is normalized to canonical form: the last partial quotient
    is always greater than 1 (every finite CF has two representations;
    this eliminates the one ending in ...1).
    """
    cf: list[int] = []
    while len(cf) < 20:  # cap at 20 terms for irrational numbers
        a = math.floor(x)  # integer part (partial quotient)
        cf.append(a)
        x = x - a  # fractional remainder
        if x < 1e-11:  # remainder negligible - exact rational reached
            break
        x = 1 / x  # reciprocal for next iteration
    # Normalize to canonical form: merge a trailing 1 into the previous term.
    # [a0,...,an-1, 1] and [a0,...,an-1+1] represent the same value;
    # the canonical form has last term > 1.
    if len(cf) >= 2 and cf[-1] == 1 and cf[-2] != 1:
        cf[-2] += 1
        cf.pop()
    return cf


# 3.245 = 649/200 -> [3; 4, 12, 4]
encode_cf(3.245)

**Decoding: recovering a value from its continued fraction.**
Given the partial quotients $[a_0; a_1, \ldots, a_n]$, the value is
recovered using the **convergent recurrence**:

$$
h_k = a_k\, h_{k-1} + h_{k-2}
\qquad
k_k = a_k\, k_{k-1} + k_{k-2}
$$

with seeds $h_{-2}=0,\; h_{-1}=1,\; k_{-2}=1,\; k_{-1}=0$.
After processing all terms the value is the rational $h_n / k_n$,
where $h_n$ and $k_n$ are the numerator and denominator of the
best rational approximation to $x$ using the given terms.

In [ ]:
# Cell 02 - Decode a continued fraction back to a floating-point value


def decode_cf(cf: list[int]) -> float:
    """Evaluate a continued fraction given as a list of partial quotients.

    Uses the standard convergent recurrence h_k = a_k * h_{k-1} + h_{k-2}
    (and identically for the denominator k_k), returning h_n / k_n.
    """
    h_prev2, h_prev = 0, 1  # h_{-2}, h_{-1}
    k_prev2, k_prev = 1, 0  # k_{-2}, k_{-1}
    for a in cf:
        h = a * h_prev + h_prev2
        k = a * k_prev + k_prev2
        h_prev2, h_prev = h_prev, h
        k_prev2, k_prev = k_prev, k
    return h_prev / k_prev


# Round-trip check: [3; 4, 12, 4] should recover 3.245 = 649/200
decode_cf([3, 4, 12, 4])

**Three types of continued fractions.**
The `eval_cf` helper below encodes a number, decodes it, and prints
both to show the round-trip.
The examples that follow illustrate the three structural types:

- **Type 1 - finite:** rational numbers terminate in a finite number of terms.
- **Type 2 - periodic:** quadratic irrationals ($\sqrt{n}$ for non-square $n$)
  produce a repeating sequence of partial quotients.
- **Type 3 - patterned:** some transcendentals like $e$ have a recognizable
  but non-repeating pattern in their partial quotients.

In [ ]:
# Cell 03 - Helper to display the full encode -> decode round-trip


def eval_cf(x: float) -> None:
    """Encode x as a continued fraction, decode it, and print the round-trip."""
    cf = encode_cf(x)
    x2 = decode_cf(cf)
    print(f"{x} -> {cf} -> {x2}")


# Type 1: rational number - produces a finite continued fraction
# 3.245 = 649/200 = [3; 4, 12, 4]  (4 terms)
eval_cf(3.245)

In [ ]:
# Cell 04 - Type 2: sqrt(2) - periodic CF with repeating sequence [1; 2, 2, 2, ...]
# The period is simply {2}, the simplest possible periodic CF for a square root

eval_cf(math.sqrt(2))

In [ ]:
# Cell 05 - Type 2: sqrt(113) - periodic CF with a longer repeating cycle
# Period is {1, 1, 1, 2, 2, 1, 1, 1, 20}, length 9

eval_cf(math.sqrt(113))

In [ ]:
# Cell 06 - Type 3: e - non-periodic but patterned CF
# e = [2; 1, 2, 1, 1, 4, 1, 1, 6, 1, 1, 8, ...]
# Pattern: every third term is 2, 4, 6, 8, ... (increasing even numbers)

eval_cf(math.e)

In [ ]:
# Cell 07 - The golden ratio phi = (1 + sqrt(5)) / 2
# phi = [1; 1, 1, 1, 1, ...] - all partial quotients are 1
# This makes phi the "most irrational" number: all-1 CF converges
# the most slowly, meaning rational approximations to phi are the
# worst possible for any irrational of its size

golden_ratio = (1 + math.sqrt(5)) / 2
eval_cf(golden_ratio)